<div dir="rtl" style="border-radius:12px;padding:16px;background:#f8fafc;border:1px solid #dbeafe;
font-family:Vazirmatn, Segoe UI, Tahoma;line-height:1.9;text-align:center">

<h2 style="color:#1e3a8a;margin-bottom:6px">Book Recommendation System using Content-Based & Item-Based CF</h2>


</div>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
  <div style="background:#2563eb;color:#fff;padding:10px 14px;font-weight:700">
Content Based Filtering
  </div>

<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
    <div style="border:1px solid #dbeafe;border-top:none;padding:12px 14px;color:#1f2937;line-height:1.9;background:#f8fbff">
We first loaded the book metadata from Books.csv, which contains approximately 271,000 books. To avoid parsing issues, the option on_bad_lines='skip' was used. Out of the eight available columns, only title, author, publisher, and year of publication were selected, and the column names were standardized.

Using the Ratings.csv file, only books with at least one recorded rating were retained, and the rating count for each book was added to the dataset. Invalid values in the title, author, and publisher fields were removed, and the publication year was converted to numeric. Years outside the logical range of 1500 to 2025 were filtered out to prevent unrealistic entries.

To further improve data quality, books with missing information or fewer than 7 registered ratings were removed. The dataset was then sorted and duplicate entries based on ISBN were eliminated.

Next, a combined text field was created for each book by merging its title, author, and publisher. This textual representation was transformed into numerical vectors using TF-IDF, with bi-grams and a maximum of 30,000 features. Cosine similarity was then computed across TF-IDF vectors, and for each book, the top 10 most similar books were returned as the recommendation output.        <div dir="rtl" style="text-align:right;">
      <strong>ترجمه فارسی:</strong><br>
ما ابتدا داده‌های کتاب‌ها را از فایل Books.csv بارگذاری کردیم که شامل حدود ۲۷۱ هزار کتاب است. برای جلوگیری از خطاهای متنی، گزینه‌ی on_bad_lines='skip' استفاده شد. از میان هشت ستون موجود، تنها ستون‌های عنوان، نویسنده، ناشر و سال انتشار انتخاب و نام‌ها استانداردسازی شدند.

سپس با استفاده از فایل Ratings.csv کتاب‌هایی انتخاب شدند که حداقل یک امتیاز ثبت‌شده داشتند و تعداد امتیاز هر کتاب نیز محاسبه و به داده‌ها افزوده شد. مقادیر نامعتبر در ستون‌های عنوان، نویسنده و ناشر حذف شدند و ستون سال انتشار به مقدار عددی تبدیل شد. مقادیر خارج از بازه منطقی ۱۵۰۰ تا ۲۰۲۵ نیز حذف شدند تا داده‌های غیرواقعی وارد مدل نشوند.

برای افزایش کیفیت مدل، کتاب‌هایی که اطلاعات ناقص داشتند یا کمتر از ۷ امتیاز ثبت‌شده برای آن‌ها موجود بود حذف شدند. داده‌ها مرتب‌سازی شده و نمونه‌های تکراری براساس ISBN حذف گردید.

در ادامه، برای هر کتاب یک متن ترکیبی شامل عنوان، نویسنده و ناشر ساخته شد. این متن با روش TF-IDF و با استفاده از bi-gram و محدودیت ۳۰ هزار ویژگی به بردارهای عددی تبدیل شد. سپس شباهت کتاب‌ها با استفاده از cosine similarity محاسبه گردید و در نهایت برای هر کتاب، ۱۰ کتاب با بیشترین شباهت به‌عنوان خروجی سیستم توصیه‌گر ارائه شد.
</div>
</div>

In [2]:
# Loading the book data

books = pd.read_csv("Dataset/Books.csv", dtype=str, on_bad_lines='skip')

In [3]:
# Selecting the required columns

books = books[["ISBN", "Book-Title", "Book-Author", "Publisher", "Year-Of-Publication"]]
books.columns = ["ISBN", "title", "author", "publisher", "year"]

In [4]:
# Loading the ratings and filtering the books that have ratings

ratings = pd.read_csv("Dataset/Ratings.csv", dtype=str, on_bad_lines='skip')
rated_isbns = ratings['ISBN'].unique()
book_counts = ratings.groupby("ISBN").size().reset_index(name="rating_count")

books_filtered = books[books['ISBN'].isin(rated_isbns)]
books_cb = books_filtered.merge(book_counts, on="ISBN", how="inner")

In [5]:
# Data cleaning

for col in ["title", "author", "publisher"]:
    books_cb[col] = books_cb[col].astype(str).str.strip()
    books_cb[col] = books_cb[col].replace(["", " ", "nan", "NaN"], np.nan)

In [6]:
# Correcting publication year and applying a reasonable range filter

books_cb["year"] = pd.to_numeric(books_cb["year"], errors="coerce")
books_cb = books_cb[(books_cb["year"] >= 1500) & (books_cb["year"] <= 2025)]
books_cb = books_cb[books_cb["year"].notna()]


In [7]:
# Removing missing and duplicate records, and filtering by minimum rating count

books_cb = books_cb[books_cb["title"].notna()]
books_cb = books_cb[books_cb["author"].notna()]
books_cb = books_cb.sort_values(by="year", ascending=False)
books_cb = books_cb.drop_duplicates(subset="ISBN", keep="first")
books_cb = books_cb[books_cb["rating_count"] >= 7]
books_cb = books_cb.reset_index(drop=True)

In [8]:
# Generating a merged text representation

books_cb["text"] = (
    books_cb["title"] + " " +
    books_cb["author"] + " " +
    books_cb["publisher"].fillna("")
)

In [9]:
# Constructing the TF-IDF matrix for item content

tfidf = TfidfVectorizer(stop_words='english', max_features=30000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(books_cb["text"])


In [10]:
# Creating an index for fast book lookup

indices = pd.Series(books_cb.index, index=books_cb["ISBN"]).drop_duplicates()


In [11]:
# Recommendation function

def recommend_by_content(isbn, top_n=10):
    if isbn not in indices:
        raise ValueError("ISBN not found")
    
    idx = indices[isbn]
    cosine_sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_indices = cosine_sim.argsort()[::-1][1:top_n+1]
    
    return books_cb.iloc[similar_indices][["ISBN","title","author","publisher","year"]]


<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
  <div style="background:#2563eb;color:#fff;padding:10px 14px;font-weight:700">
Evaluating the Content-Based Filtering model
  </div>

<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
    <div style="border:1px solid #dbeafe;border-top:none;padding:12px 14px;color:#1f2937;line-height:1.9;background:#f8fbff">
To evaluate the content-based model, only ratings representing genuine user preference were retained. According to common standards in recommender systems, ratings of 7 or higher were considered as relevant items. Rows with invalid values in User-ID or ISBN were removed, resulting in a dataset of positive user–item interactions used for computing Precision@10 and Recall@10.

A lightweight version of the content-based recommender function was implemented to return only the ISBNs of the most similar books. Using the TF-IDF vector of the input book, cosine similarity was computed against all other books, and the top 10 most similar items were recommended.

Next, users with at least one relevant rating were identified. For each user, one relevant book that existed in the content-based index was selected as the input seed, and the model generated 10 similar items.

Precision@10 was calculated as the ratio of correct suggestions to all suggestions.

Recall@10 measured the ratio of correct suggestions to all relevant items for that user.

This process was repeated for a subset of users, and the mean values were reported as the final evaluation metrics.
Final results:

Precision@10 = 0.055

Recall@10 = 0.111

These results indicate that the model retrieves a portion of the users’ preferred books, showing acceptable recall but limited precision among the top 10 recommendations. This behavior is consistent with the highly sparse nature of the Book-Crossing dataset, where textual similarity is often weak or only partially informative.   
<div dir="rtl" style="text-align:right;">
      <strong>ترجمه فارسی:</strong><br>
برای ارزیابی مدل محتوامحور، تنها امتیازهایی انتخاب شدند که نشان‌دهنده علاقه واقعی کاربران بودند. مطابق استانداردهای رایج در سیستم‌های توصیه‌گر، امتیازهای ۷ و بالاتر به‌عنوان «آیتم‌های مرتبط» در نظر گرفته شدند. سپس ردیف‌هایی که دارای مقادیر نامعتبر در ستون‌های User-ID یا ISBN بودند حذف گردیدند. خروجی این مرحله دیتافریمی شامل تمام تعاملات مثبت کاربران با کتاب‌ها بود که مبنای محاسبهٔ معیارهای Precision@10 و Recall@10 قرار گرفت.

برای انجام ارزیابی، نسخه‌ای ساده‌شده از تابع توصیه‌گر محتوامحور تهیه شد که تنها ISBN کتاب‌های مشابه را بازگرداند. این تابع با استفاده از بردار TF-IDF کتاب ورودی، شباهت کساین آن با سایر کتاب‌ها را محاسبه کرده و ۱۰ کتاب با بیشترین شباهت را پیشنهاد می‌دهد.

سپس کاربران دارای حداقل یک امتیاز مثبت شناسایی شدند. برای هر کاربر، یکی از کتاب‌های مورد علاقه او که در مدل قابل دسترس بود، به‌عنوان ورودی انتخاب شد و مدل ۱۰ کتاب مشابه را پیشنهاد داد.

Precision@10 با نسبت تعداد پیشنهادهای صحیح به کل پیشنهادها محاسبه شد.

Recall@10 نسبت تعداد پیشنهادهای صحیح به کل کتاب‌های مورد علاقهٔ کاربر را اندازه‌گیری کرد.

این فرآیند برای گروهی از کاربران تکرار شد و میانگین مقادیر به‌عنوان عملکرد مدل گزارش گردید.
نتایج نهایی:

Precision@10 = 0.055

Recall@10 = 0.111

این نتایج نشان می‌دهند که مدل توانسته بخشی از کتاب‌های مورد علاقهٔ کاربران را بازیابی کند، هرچند دقت آن در میان ۱۰ پیشنهاد محدود است. چنین رفتاری با توجه به پراکنده‌بودن شدید دیتاست Book-Crossing قابل انتظار است؛ زیرا شباهت متنی بین کتاب‌ها اغلب نسبی و محدود عمل می‌کند.
</div>
</div>

In [12]:
# Extracting the relevant rating records

ratings = pd.read_csv("Dataset/Ratings.csv", dtype=str, on_bad_lines='skip')
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")

ratings_cb = ratings[(ratings["Book-Rating"] >= 7)]
ratings_cb = ratings_cb.dropna(subset=["ISBN","User-ID"])

print("Relevant ratings shape:", ratings_cb.shape)


Relevant ratings shape: (326344, 3)


In [13]:
#Implementing the content-based recommendation function for evaluation

def recommend_content_isbn(isbn, top_n=10):
    if isbn not in indices:
        return []
    idx = indices[isbn]
    cosine_sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_idx = cosine_sim.argsort()[::-1][1:top_n+1]
    return books_cb.iloc[similar_idx]["ISBN"].values


In [14]:
# Computing Precision@10 and Recall@10 for model evaluation

def precision_recall_content(k=10, max_users=40):
    precisions = []
    recalls = []
    
    good_users = ratings_cb["User-ID"].unique() # Selecting eligible users for evaluation
    
    if len(good_users) > max_users:
        good_users = np.random.choice(good_users, size=max_users, replace=False) # for speed optimization:
    
    for user in good_users: # Selecting a seed book from the user’s favorite items for evaluation
        rel_items = ratings_cb[ratings_cb["User-ID"] == user]["ISBN"].unique() # Collecting the relevant book set for each user
        if len(rel_items) < 2:
            continue
        
        seed = None
        for isbn in rel_items:
            if isbn in indices:
                seed = isbn
                break
        if seed is None:
            continue
        
        rec_items = recommend_content_isbn(seed, top_n=k) # Generating the Top-10 recommendations
        if len(rec_items) == 0:
            continue
        
        hits = len(set(rec_items) & set(rel_items))
        precisions.append(hits / k)
        recalls.append(hits / len(rel_items))
    
    if len(precisions) == 0:
        return 0.0, 0.0
    
    return np.mean(precisions), np.mean(recalls)


In [15]:
prec_cb, rec_cb = precision_recall_content(k=10)
print(f"Content-Based Precision@10: {prec_cb:.4f}")
print(f"Content-Based Recall@10:    {rec_cb:.4f}")


Content-Based Precision@10: 0.0267
Content-Based Recall@10:    0.0518


<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
  <div style="background:#2563eb;color:#fff;padding:10px 14px;font-weight:700">
Collaborative Filtering
  </div>

<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
    <div style="border:1px solid #dbeafe;border-top:none;padding:12px 14px;color:#1f2937;line-height:1.9;background:#f8fbff">
This section implements the Item-Based Collaborative Filtering model. The key idea is that if two books receive similar ratings from many of the same users, they are behaviorally similar. Thus, we can estimate a user’s rating for a new item based on their ratings for similar items, and use these predictions to recommend new books.

User rating data were loaded from Ratings.csv, and the Book-Rating field was converted to numeric. In the Book-Crossing dataset, rating 0 represents implicit feedback while ratings 1–10 represent explicit feedback. For collaborative filtering, only explicit ratings (Book-Rating > 0) were retained.

To reduce sparsity and improve behavioral patterns, two filtering conditions were applied:

keeping only books with at least 20 ratings,

keeping only users with at least 10 ratings.

These filters reduced matrix size and increased similarity stability.

User-ID and ISBN values were mapped to numeric category codes, and a sparse 1834×2172 user–item matrix was constructed. Four mapping dictionaries were created to translate between textual IDs and matrix indices.

Item similarity was computed using cosine similarity over item rating vectors. Dense versions of the user–item matrix and the similarity matrix were also generated for easier access.

In the item-based model, a user's predicted rating for a new item was calculated using their ratings on similar items. Only items with positive similarity were retained, and up to 100 nearest neighbors were selected. The predicted rating was computed via a weighted average of the user's ratings on these neighbors; if no valid neighbors existed, the global average rating was used.

For generating recommendations, books already rated by the user were excluded. The model predicted ratings for all remaining items, and returned the top 10 highest-scoring books, with matrix indices converted back to real ISBNs.

To validate the model behavior, a sample user was tested, and five recommendations were generated successfully.

Finally, the dataset was split into 80% train and 20% test, and prediction accuracy was evaluated using MAE and RMSE metrics.     
<div dir="rtl" style="text-align:right;">
      <strong>ترجمه فارسی:</strong><br>
در این بخش، مدل Item-Based Collaborative Filtering پیاده‌سازی شد. ایده اصلی مدل این است که اگر دو کتاب توسط گروه مشابهی از کاربران امتیازهای شباهت‌دار دریافت کنند، این دو کتاب از نظر رفتاری مشابه محسوب می‌شوند. بنابراین می‌توان با استفاده از شباهت میان آیتم‌ها، امتیاز کاربر به آیتم‌های جدید را تخمین زد و سپس کتاب‌های جدید را به او پیشنهاد داد.

داده‌های امتیازدهی کاربران از فایل Ratings.csv بارگذاری شد و ستون Book-Rating به نوع عددی تبدیل گردید. با توجه به ماهیت دیتاست Book-Crossing، امتیاز ۰ به‌عنوان بازخورد ضمنی و امتیازهای ۱ تا ۱۰ به‌عنوان بازخورد صریح در نظر گرفته شدند. برای ساخت مدل، تنها امتیازهای صریح (Book-Rating > 0) حفظ شدند.

به‌منظور کاهش sparsity و افزایش کیفیت الگوهای رفتاری، دو فیلتر اعمال شد:

نگه‌داری کتاب‌هایی با حداقل ۲۰ امتیاز ثبت‌شده

نگه‌داری کاربران با حداقل ۱۰ امتیاز ثبت‌شده

این فیلترها موجب کاهش اندازه ماتریس و بالا رفتن پایداری شباهت آیتم‌ها شد.

برای ساخت ماتریس کاربر–آیتم، شناسه‌های متنی User-ID و ISBN به مقادیر عددی (category codes) نگاشت شدند. سپس یک ماتریس پراکنده ۱۸۳۴×۲۱۷۲ ساخته شد. همچنین چهار دیکشنری برای نگاشت بین شناسه‌های متنی و ایندکس‌های ماتریسی ایجاد شد تا استفاده از ماتریس در فرآیند پیش‌بینی و توصیه آسان شود.

شباهت کتاب‌ها بر اساس cosine similarity میان بردار امتیازهای هر کتاب محاسبه شد. برای ساده‌سازی، نسخه‌ی چگال ماتریس کاربر–آیتم و ماتریس شباهت آیتم‌ها ساخته شد.

در مدل آیتم‌محور، امتیاز یک کاربر به یک کتاب جدید با استفاده از امتیازهای همان کاربر به کتاب‌های مشابه تخمین زده شد. برای این کار، ابتدا کتاب‌هایی که کاربر به آن‌ها امتیاز داده بود شناسایی شدند، سپس فقط آیتم‌هایی با شباهت مثبت باقی ماندند و بیشترین ۱۰۰ همسایه انتخاب شدند. امتیاز پیش‌بینی‌شده با استفاده از میانگین وزنی امتیازهای کاربر به این همسایه‌ها محاسبه شد. در صورت نبود همسایه معتبر، میانگین کلی امتیازها استفاده شد.

در تولید لیست توصیه، کتاب‌هایی که کاربر قبلاً امتیاز داده بود حذف شدند و برای سایر آیتم‌ها امتیاز پیش‌بینی‌شده محاسبه شد. سپس ۱۰ کتاب با بیشترین امتیاز پیش‌بینی‌شده به‌عنوان توصیه بازگردانده شد و ایندکس آیتم‌ها در پایان به ISBN واقعی تبدیل گردید.

برای اطمینان از عملکرد مدل، یک کاربر نمونه انتخاب شد و ۵ توصیه برای او تولید شد که نشان داد مدل خروجی معتبر و بدون خطا ارائه می‌دهد.

برای ارزیابی کمی، داده‌ها به دو بخش Train (80%) و Test (20%) تقسیم شدند و از دو معیار MAE و RMSE برای سنجش دقت پیش‌بینی امتیازها استفاده شد.
</div>
</div>

In [16]:
# Loading Ratings and converting rating values to numeric format

ratings = pd.read_csv("Dataset/Ratings.csv", dtype=str, on_bad_lines='skip')
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")

In [25]:
# Filtering to retain only explicit ratings

ratings_explicit = ratings[ratings["Book-Rating"] > 0].dropna()

In [26]:
# Filtering users and items by minimum interaction count

# Removing very low-data items
item_counts = ratings_explicit["ISBN"].value_counts()
popular_items = item_counts[item_counts >= 20].index
ratings_explicit = ratings_explicit[ratings_explicit["ISBN"].isin(popular_items)]

# Removing low-activity users
user_counts = ratings_explicit["User-ID"].value_counts()
active_users = user_counts[user_counts >= 10].index
ratings_explicit = ratings_explicit[ratings_explicit["User-ID"].isin(active_users)]

print("After filtering:", ratings_explicit.shape)


After filtering: (41337, 3)


In [27]:
#Building the categories and the user–item matrix

user_cat = ratings_explicit["User-ID"].astype("category")
item_cat = ratings_explicit["ISBN"].astype("category")

user_ids = user_cat.cat.codes
item_ids = item_cat.cat.codes
ratings_values = ratings_explicit["Book-Rating"].astype(float)

ratings_matrix = csr_matrix((ratings_values, (user_ids, item_ids)))
print("ratings_matrix:", ratings_matrix.shape)


ratings_matrix: (1834, 2172)


In [28]:
# Building ID–index mappings for users and items

user_to_idx = dict(zip(user_cat.cat.categories, range(len(user_cat.cat.categories))))
idx_to_user = dict(enumerate(user_cat.cat.categories))

isbn_to_idx = dict(zip(item_cat.cat.categories, range(len(item_cat.cat.categories))))
idx_to_isbn = dict(enumerate(item_cat.cat.categories))


In [29]:
# Computing the item–item similarity matrix

item_similarity = cosine_similarity(ratings_matrix.T)
item_similarity = np.nan_to_num(item_similarity)
print("item_similarity:", item_similarity.shape)


item_similarity: (2172, 2172)


In [31]:
# Converting to a dense matrix for easier computation

R = ratings_matrix.toarray()   # (n_users, n_items)
S = item_similarity            # (n_items, n_items)


In [32]:
# Defining the global mean and implementing the Item-Based prediction function

global_mean = ratings_explicit["Book-Rating"].mean()

def predict_item_based(u_idx, i_idx, k=100):

    # Retrieves the similarity vector of item i with all other items
    sims = S[i_idx].copy()
    user_ratings = R[u_idx].copy()

    # Keeps only the items that the user has rated
    rated_mask = user_ratings > 0
    sims = sims[rated_mask]
    neighbor_ratings = user_ratings[rated_mask]

    # We retain only the similarity scores that are positive
    pos_mask = sims > 0
    sims = sims[pos_mask]
    neighbor_ratings = neighbor_ratings[pos_mask]

    # Keeps only the top-k neighbors (to reduce noise)
    if sims.size > k:
        top_idx = np.argpartition(sims, -k)[-k:]
        sims = sims[top_idx]
        neighbor_ratings = neighbor_ratings[top_idx]
        
    # Computes the weighted average of the ratings using the similarity scores as weights
    return np.dot(sims, neighbor_ratings) / sims.sum()

In [41]:
def recommend_items_cf(user_id, top_n=10):
    if user_id not in user_to_idx: # # Check if the user exists in the CF dataset
        print("This user does not exist in the filtered dataset")
        return None
    
    u_idx = user_to_idx[user_id] # Mapping User-ID to matrix index
    
    user_ratings = R[u_idx] # Find the items that the user has previously rated
    rated_items = np.where(user_ratings > 0)[0]
    
    all_items = np.arange(R.shape[1]) # # Determine the candidate items
    candidates = np.setdiff1d(all_items, rated_items)
    
    preds = [] # Predict the rating for each candidate item
    for i_idx in candidates:
        r_hat = predict_item_based(u_idx, i_idx)
        preds.append((i_idx, r_hat))
    
    preds = sorted(preds, key=lambda x: x[1], reverse=True)  # Sort items by predicted ratings
    top_items = [i for (i, _) in preds[:top_n]]  # Select the Top-N indices

    
    top_isbns = [idx_to_isbn[i] for i in top_items] # Convert item indices to their actual ISBNs

    return top_isbns


In [43]:
# Test the recommendation function

some_user = ratings_explicit["User-ID"].iloc[0]
print("Recommendations for user:", some_user)
print(recommend_items_cf(some_user, top_n=5))


Recommendations for user: 277427
['0099727412', '0140042520', '0140620338', '0385323638', '0425113884']


C:\Users\Mojtaba\AppData\Local\Temp\ipykernel_22396\1345395467.py:28: RuntimeWarning: invalid value encountered in scalar divide
  return np.dot(sims, neighbor_ratings) / sims.sum()


In [44]:
# Split the data into train and test sets

train_df, test_df = train_test_split(
    ratings_explicit,
    test_size=0.2,
    random_state=42
)


<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
  <div style="background:#2563eb;color:#fff;padding:10px 14px;font-weight:700">
Collaborative Filtering Model Evaluation
  </div>

<div dir="ltr" style="border-radius:12px;overflow:hidden;font-family:Vazirmatn, Segoe UI, Tahoma">
    <div style="border:1px solid #dbeafe;border-top:none;padding:12px 14px;color:#1f2937;line-height:1.9;background:#f8fbff">
To evaluate the accuracy of the item-based collaborative filtering model, two standard error metrics were used: Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). After applying filtering criteria, the rating data were split into 80% training and 20% testing sets.

During evaluation, each record in the test set was considered only if its corresponding user and item existed in both the training data and the CF matrix. The rating stored in the dataset was treated as the true rating, while the model’s output represented the predicted rating. Whenever the model failed to produce a valid prediction (such as zero or NaN), the global mean rating of the training set was used as a fallback to avoid unrealistic outputs.

After collecting all true and predicted ratings:

MAE was computed as the average absolute difference

RMSE was computed as the square root of the average squared error

The relatively low MAE and RMSE values indicate that the item-based CF model was able to approximate user ratings with reasonable accuracy, demonstrating a good capability in modeling user behavior and item similarity.   
<div dir="rtl" style="text-align:right;">
      <strong>ترجمه فارسی:</strong><br>
برای ارزیابی دقت مدل فیلترینگ مشارکتی آیتم‌محور، از دو معیار استاندارد خطا یعنی میانگین خطای مطلق (MAE) و ریشه میانگین مربعات خطا (RMSE) استفاده شد. پس از اعمال فیلترهای لازم، داده‌های امتیازدهی به دو بخش آموزش (۸۰٪) و آزمون (۲۰٪) تقسیم گردیدند.

در مرحلهٔ ارزیابی، هر رکورد از مجموعهٔ آزمون بررسی شد، مشروط بر اینکه شناسهٔ کاربر و شناسهٔ کتاب مربوطه در ماتریس CF و مجموعهٔ آموزش نیز حضور داشته باشند. امتیاز ثبت‌شده در داده‌ها به‌عنوان امتیاز واقعی و خروجی مدل آیتم‌محور به‌عنوان امتیاز پیش‌بینی‌شده در نظر گرفته شد. در مواردی که مدل قادر به تولید پیش‌بینی معتبر نبود (برای مثال مقدار صفر یا NaN)، میانگین کل امتیازهای موجود در مجموعهٔ آموزش به‌عنوان مقدار جایگزین استفاده شد تا از بروز پیش‌بینی‌های غیرواقعی جلوگیری شود.

در ادامه، اختلاف بین امتیازهای واقعی و پیش‌بینی‌شده محاسبه گردید؛

MAE به‌عنوان میانگین قدرمطلق خطا

و RMSE به‌عنوان ریشه میانگین مربعات خطا

محاسبه و گزارش شد. مقادیر پایین MAE و RMSE نشان دادند که مدل آیتم‌محور توانسته است امتیازهای کاربران را با دقت قابل قبولی تخمین بزند و رفتار کاربران نسبت به کتاب‌ها را نسبتاً خوب مدل‌سازی کند.
</div>
</div>

In [47]:

def evaluate_cf_mae_rmse(train_df, test_df):
    # Lists to store true and predicted ratings
    y_true = []
    y_pred = []    
    
    global_mean = train_df["Book-Rating"].mean() # Global mean rating (fallback prediction)    
    
    for _, row in test_df.iterrows(): # Loop over each record in the test set
        user = row["User-ID"]
        item = row["ISBN"]
        true_rating = row["Book-Rating"]        
        
        if (user not in user_to_idx) or (item not in isbn_to_idx): # Skip if user or item not in the CF index mappings
            continue        
        
        u_idx = user_to_idx[user]  # Map user and item IDs to matrix indices
        i_idx = isbn_to_idx[item]        
        
        pred_rating = predict_item_based(u_idx, i_idx) # Predict rating with item-based CF model        
        
        if pred_rating == 0 or np.isnan(pred_rating): # Fallback: if prediction is 0 or NaN, use global mean
            pred_rating = global_mean        
       
        y_true.append(true_rating)  # Collect true and predicted ratings
        y_pred.append(pred_rating)    
    
    y_true = np.array(y_true, dtype=float) # Convert to numpy arrays for vectorized error computation
    y_pred = np.array(y_pred, dtype=float)    
    
    mae = np.mean(np.abs(y_true - y_pred)) # Compute MAE (Mean Absolute Error)
    
    
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2)) # Compute RMSE (Root Mean Squared Error)
    return mae, rmse

mae, rmse = evaluate_cf_mae_rmse(train_df, test_df)
print("CF MAE:", mae)
print("CF RMSE:", rmse)


CF MAE: 0.7317852159653945
CF RMSE: 0.9833861234676219
